# PDQ Attack Visualization

PDQ-only visualization notebook for the isolated subset attacks in `fdeph_eval/attacks/pdq/`.

This notebook does three things:
1. Loads one PDQ run log and summarizes per-image progress.
2. Finds saved adversarial outputs for successful attacks.
3. Renders image panels with a 16x16 PDQ hash grid, adversarial hash grid, and XOR grid.

It is designed for the current subset workflow first, not cross-method comparison.
            


In [ ]:
import os
import sys
from pathlib import Path

HERE = Path(os.getcwd())
REPO_ROOT = HERE
if not (REPO_ROOT / 'fdeph_eval').exists():
    REPO_ROOT = HERE.parent.parent if (HERE.parent.parent / 'fdeph_eval').exists() else HERE

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

print('Repo root:', REPO_ROOT)
            


In [ ]:
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
from PIL import Image

from utils.pdq_torch import compute_pdq_hard
from utils.image_processing import load_and_preprocess_img
from fdeph_eval.analysis.plotting import load_attack_csv

ATTACK_VARIANT = 'whitebox'  # 'whitebox' or 'blackbox'
THRESHOLD = '0.08'
SUBSET_DIR = REPO_ROOT / 'inputs' / 'pdq_subset_10'
FIG_DIR = REPO_ROOT / 'fdeph_eval' / 'analysis' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = REPO_ROOT / 'logs' / f'attack_steps_pdq_{ATTACK_VARIANT}_subset10_T{THRESHOLD}.csv'
OUT_DIR = REPO_ROOT / f'evasion_attack_outputs_pdq_{ATTACK_VARIANT}_T{THRESHOLD}_subset10'

N_SHOW = 5
SAVE_FIG = True
FIG_BASENAME = f'pdq_attack_visualization_{ATTACK_VARIANT}_T{THRESHOLD.replace('.', '')}'

device = 'cpu'
print('Log:', LOG_PATH)
print('Outputs:', OUT_DIR)
print('Subset:', SUBSET_DIR)
            


In [ ]:
df = load_attack_csv(LOG_PATH)
print('Rows:', len(df))
print('Unique images:', df['image_id'].nunique())

per_image = (
    df.sort_values(['image_id', 'step'])
      .groupby('image_id', as_index=False)
      .agg(
          max_dist_norm=('dist_norm', 'max'),
          max_dist_raw=('dist_raw', 'max'),
          best_step=('step', 'max'),
          min_l2=('l2', 'min'),
          min_linf=('linf', 'min'),
          max_success=('success', 'max'),
      )
      .sort_values(['max_success', 'max_dist_norm'], ascending=[False, False])
)

success_count = int(per_image['max_success'].sum())
print('Images with success:', success_count)
display(per_image.head(10))
            


In [ ]:
plt.figure(figsize=(8, 4))
plt.hist(per_image['max_dist_norm'], bins=10)
plt.xlabel('Best normalized PDQ distance reached')
plt.ylabel('Number of images')
plt.title(f'PDQ {ATTACK_VARIANT} subset run: best distance per image')
plt.show()
            


In [ ]:
def find_original(image_id: str) -> Path | None:
    for ext in ['.JPEG', '.jpeg', '.jpg', '.png', '.PNG']:
        p = SUBSET_DIR / f'{image_id}{ext}'
        if p.exists():
            return p
    return None


def find_adversarial(image_id: str) -> Path | None:
    for ext in ['.png', '.PNG', '.jpeg', '.jpg', '.JPEG']:
        p = OUT_DIR / f'{image_id}_opt_pdq{ext}'
        if p.exists():
            return p
    for ext in ['.png', '.PNG', '.jpeg', '.jpg', '.JPEG']:
        p = OUT_DIR / f'{image_id}_opt{ext}'
        if p.exists():
            return p
    return None


def load_rgb(path: Path) -> np.ndarray:
    return np.array(Image.open(path).convert('RGB')).astype(np.float32) / 255.0


def draw_hash_grid(ax, grid, title, flip_mask=None):
    ax.set_xlim(0, 16)
    ax.set_ylim(0, 16)
    ax.set_aspect('equal')
    ax.axis('off')
    ax.set_title(title, fontsize=9, pad=4)
    for r in range(16):
        for c in range(16):
            bit = int(grid[r, c])
            if flip_mask is not None and flip_mask[r, c]:
                color, edge, lw = '#E24B4A', '#A32D2D', 0.8
            elif bit == 1:
                color, edge, lw = '#378ADD', '#185FA5', 0.3
            else:
                color, edge, lw = '#f0f0f0', '#cccccc', 0.3
            ax.add_patch(
                patches.Rectangle((c, 15 - r), 1, 1, linewidth=lw, edgecolor=edge, facecolor=color)
            )


def compute_pdq_grid(path: Path):
    tensor = load_and_preprocess_img(str(path), device, resize=True)
    hard_hash, _, _ = compute_pdq_hard(tensor)
    bits = hard_hash.cpu().numpy().astype(int).reshape(16, 16)
    return bits
            


In [ ]:
paired = []
for row in per_image.to_dict(orient='records'):
    image_id = row['image_id']
    ori = find_original(image_id)
    adv = find_adversarial(image_id)
    if ori is not None and adv is not None:
        paired.append({**row, 'ori_path': ori, 'adv_path': adv})

print('Saved adversarial pairs found:', len(paired))
if not paired:
    print('No saved adversarial outputs found in the configured output folder.')
    print('This usually means the run has not produced any success images yet, or a different output folder/threshold is needed.')
else:
    display(pd.DataFrame(paired)[['image_id', 'max_dist_norm', 'max_dist_raw', 'max_success', 'ori_path', 'adv_path']].head(N_SHOW))
            


In [ ]:
if paired:
    show_rows = paired[:N_SHOW]
    n_rows = len(show_rows)
    fig, axes = plt.subplots(n_rows, 6, figsize=(18, 3.2 * n_rows))
    if n_rows == 1:
        axes = np.expand_dims(axes, axis=0)

    for row_idx, item in enumerate(show_rows):
        ori = load_rgb(item['ori_path'])
        adv = load_rgb(item['adv_path'])
        diff = np.clip(np.abs(adv - ori) * 10.0, 0.0, 1.0)

        ori_grid = compute_pdq_grid(item['ori_path'])
        adv_grid = compute_pdq_grid(item['adv_path'])
        xor_grid = (ori_grid != adv_grid).astype(int)

        axes[row_idx, 0].imshow(ori)
        axes[row_idx, 0].axis('off')
        axes[row_idx, 0].set_title(f'Original\n{item["image_id"]}', fontsize=9)

        axes[row_idx, 1].imshow(adv)
        axes[row_idx, 1].axis('off')
        axes[row_idx, 1].set_title(
            f'Adversarial\nd={item["max_dist_raw"]:.0f} ({item["max_dist_norm"]:.4f})',
            fontsize=9,
        )

        axes[row_idx, 2].imshow(diff)
        axes[row_idx, 2].axis('off')
        axes[row_idx, 2].set_title('Difference x10', fontsize=9)

        draw_hash_grid(axes[row_idx, 3], ori_grid, 'Original hash')
        draw_hash_grid(axes[row_idx, 4], adv_grid, 'Adversarial hash', flip_mask=xor_grid)
        draw_hash_grid(axes[row_idx, 5], xor_grid, 'XOR (red = flipped)', flip_mask=xor_grid)

    plt.tight_layout()
    if SAVE_FIG:
        out_path = FIG_DIR / f'{FIG_BASENAME}.png'
        plt.savefig(out_path, dpi=220, bbox_inches='tight')
        print('Saved figure to', out_path)
    plt.show()


In [ ]:
# Optional: inspect the strongest non-successful images even if no adversarial image was saved.
# This is useful when a pilot run moved the PDQ distance but did not cross the threshold.

display(per_image[['image_id', 'max_dist_norm', 'max_dist_raw', 'max_success']].head(10))
            
